In [ ]:
# Install required packages
!pip install -q zarr segmentation-models-pytorch

In [ ]:
from torch.utils.data import Dataset,DataLoader
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import WeightedRandomSampler
import torch.nn.functional as F
from tqdm.notebook import tqdm
import cv2
import torch
import random
from torch.utils.data import Sampler
import os
import zarr
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import torch.nn as nn
import copy
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from skimage.morphology import skeletonize
import seaborn as sns
import json
import matplotlib.patches as mpatches
import datetime
import shutil
import plotly.graph_objects as go

from albumentations.core.transforms_interface import ImageOnlyTransform
from skimage.filters import frangi , hessian , meijering
from torchvision.models.swin_transformer import (
swin_t,swin_s,swin_b,
swin_v2_t,swin_v2_s,swin_v2_b,
Swin_S_Weights,Swin_V2_S_Weights,
Swin_B_Weights,Swin_V2_B_Weights,
Swin_T_Weights,Swin_V2_T_Weights
)
from transformers import AutoModelForSemanticSegmentation
from torchvision import models
from torchvision.models import VGG16_Weights
from torchvision.ops.focal_loss import sigmoid_focal_loss
from monai.losses import FocalLoss
from sklearn.metrics import accuracy_score

In [ ]:
#!/usr/bin/env python
# coding: utf-8

# In[1]:
import torch
# torch.autograd.set_detect_anomaly(True)
from torch.utils.data import DataLoader,WeightedRandomSampler
from torch.optim.lr_scheduler import PolynomialLR
import torch.nn.functional as F
from torchvision.transforms import v2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from tqdm.notebook import tqdm
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random
import os

# dataset

In [ ]:

class TrainUnetDataset(Dataset):
    def __init__(self, transform,data,normalizer,example_phase=False):
        super(TrainUnetDataset, self).__init__()
        self.data = data
        self.transform = transform
        self.normalizer = normalizer
        self.example_phase = example_phase
    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        """
        img : (H,W,C)
        maks : (H,W)
        """
        img, mask,_ = self.data[index]
        
        if img.ndim == 2: img = img[..., None]
        if mask.ndim == 2: mask = mask[..., None]

        result = self.transform(image=img, mask=mask)
        new_image_not_normal  = result['image']
        new_mask= result['mask']

        new_image = self.normalizer(image = new_image_not_normal)["image"]

        if torch.is_tensor(new_image): new_image = new_image.numpy()
        if torch.is_tensor(new_mask): new_mask = new_mask.numpy()
        
        # turn image shape to : (C,H,W)
        if new_image.ndim == 3 : 
            new_image = new_image.transpose(2, 0, 1) 
        elif new_image.ndim == 2:
            new_image = new_image[None, :, :] 
        # mask shape to : (H,W)
        if new_mask.ndim == 3: new_mask = new_mask.squeeze(-1)

        binary_mask = (new_mask !=0).astype(int)
        if(self.example_phase):
            return new_image_not_normal,new_image, binary_mask, new_mask
        
        return new_image, binary_mask, new_mask

class UnetExampleDataset(Dataset):
    def __init__(self,transform,data,base_transform=None):
        super(UnetExampleDataset,self).__init__()
        
        self.data = data
        self.transform = transform
        self.to_tensor = ToTensorV2()
        if(base_transform is None):
            self.base_transform = A.Compose([ToTensorV2()])
        else:
            self.base_transform = base_transform
    def __len__(self):
        return len(self.data)
    def __getitem__(self,index):
        img , mask = self.data[index]
        img = np.expand_dims(img, axis=-1) 
        result = self.transform(image=img, mask=mask)
        new_image = result['image']
        new_mask = result['mask']
        
        raw_result = self.base_transform(image=img, mask=mask)
        raw_image = raw_result['image']
        raw_mask = raw_result['mask']

        new_image = self.to_tensor(image = new_image)["image"]
        return new_image.float() , new_mask , raw_image.float() , raw_mask

# helpers

In [ ]:
def read_images(base_path, part,preprocessor,max_workers=None,chosen_labels = None):
    base_path = Path(base_path)
    images_base = base_path / "images" / part
    labels_base = base_path / "labels" / part

    image_names = sorted([p.name for p in os.scandir(images_base) if p.is_file()])
    if(not preprocessor):
        print("NOTE : preprocessor is not defined . no preprocessing will be used !")
    def _read_one(fname):
        name_stem = Path(fname).stem
        img_path = images_base / fname
        label_path = labels_base / f"{name_stem}.zarr"
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if(preprocessor):
            img = preprocessor(img)
        label = zarr.load(str(label_path))

        return img, label , name_stem

    if max_workers is None:
        cpu = os.cpu_count() or 4
        max_workers = min(32, cpu * 4)

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for img, label , name_stem in tqdm(ex.map(_read_one, image_names), total=len(image_names)):
            results.append([img,label,name_stem])

    return results

@torch.no_grad()
def make_example_datasets(valid_images,image_names,transform,normalizer):
    img_dict = {"easy":[],"normal":[],"hard":[],"very hard":[]}
    for img, label , name_stem in valid_images : 
        for diff in image_names:
            if(name_stem in image_names[diff]):
                img_dict[diff].append([img , label , name_stem])
                break
    for diff , images in img_dict.items():
        ds = TrainUnetDataset(
            transform = transform,
            data = images,
            example_phase=True,
            normalizer = normalizer
        )
        dl = DataLoader(
            ds,
            batch_size=1,
            num_workers=0,
            shuffle=False,
        )
        img_dict[diff] = dl

    return img_dict



def crop_dims(target , current):
    left = (current.shape[3]-target.shape[3])//2
    right = (current.shape[3]-target.shape[3]) - left
    top = (current.shape[2]-target.shape[2])//2
    down = (current.shape[2]-target.shape[2]) - top
    croped = current[:,:,top:-down , left:-right]
    return croped
def padd_dims(target , current):
    pad_h = target.shape[2] - current.shape[2] 
    pad_w = target.shape[3] - current.shape[3]
    padded = F.pad(current, (0, pad_w, 0, pad_h), mode='constant', value=0)
    return padded


@torch.no_grad()
def TP_TN_FP_FN(preds,gt,process_preds=False,return_TN=False):
    if(process_preds):
        preds_argmax = torch.argmax(preds,dim=1)
        onehot_preds = F.one_hot(preds_argmax,num_classes=preds.shape[1])
        pred_onehot = onehot_preds.permute(0, 3, 1, 2).float()
    else :
        pred_onehot = preds
        
    onehot_gt = F.one_hot(gt,num_classes=preds.shape[1])
    onehot_gt = onehot_gt.permute(0, 3, 1, 2).float()
    TN = 0
    if(return_TN):
        TN = (((1-onehot_gt)*(1-pred_onehot)).sum(dim=(0,2,3))).cpu()

    TP = ((onehot_gt*pred_onehot).sum(dim=(0,2,3))).cpu()
    FP = (((1-onehot_gt)*pred_onehot).sum(dim=(0,2,3))).cpu()
    FN = ((onehot_gt*(1-pred_onehot)).sum(dim=(0,2,3))).cpu()
    return TP , TN , FP , FN

def draw_mask(image,mask,args=None,colors=None):
    img = image.copy().astype(np.uint8)
    m = mask.astype(np.int64)
    if(colors is None):
        colors = np.array([(0,255,0)]*25,dtype=np.uint8)
    img[m>0] = colors[m[m>0]-1]
    return img

@torch.no_grad()
def plot_some_images(data,transforms,image_counts=36,fig_shape=(6,6),base_transforms=None):
    ds = UnetExampleDataset(transform=transforms , data=data,base_transform=base_transforms)
    dataloader = DataLoader(
        ds,
        batch_size = 2 ,
        num_workers = 4 ,
        pin_memory=False,
        shuffle=True
    )

    iter_loader = iter(dataloader)
    w,h=fig_shape
    plt.figure(figsize=(w*5,h*5))
    for i in range(1,image_counts+1,2):
        new_imgs , new_mask , old_imgs , old_mask = next(iter_loader)
        new_img = new_imgs[0].numpy()
        old_img = old_imgs[0].numpy()
        if(new_img.shape[0]==1):
            new_img = new_img[0]
            old_img = old_img[0]

        x_disp = (new_img- new_img.min()) / (new_img.max() - new_img.min() + 1e-8)
        new_img = np.repeat(x_disp[..., None], 3, axis=2)*255
        new_img = draw_mask(new_img,new_mask[0].numpy())

        plt.subplot(w,h,i)
        plt.imshow(old_img,cmap="gray")
        plt.title("Old Image")

        plt.subplot(w,h,i+1)
        plt.imshow(new_img)
        plt.title("New Image")

def pre_hard_skeletonize(base_path,output_path):
    parts = ["train","val","test"]
    os.makedirs(os.path.join(output_path , "skels"),exist_ok=True)
    for part in parts:
        mask_base_path = os.path.join(base_path,"labels",part)
        os.makedirs(os.path.join(output_path , "skels",part),exist_ok=True)

        mask_list = os.listdir(mask_base_path)
        for mask_name in tqdm(mask_list):
            name = Path(mask_name).stem
            mask_path = os.path.join(mask_base_path,mask_name)

            mask = zarr.load(str(mask_path))
       
            mask = (mask!=0).astype(np.uint8)
        
            out_skel_path = os.path.join(output_path,"skels",part,f"{name}.png")
            skel = skeletonize(mask).astype(np.uint8) * 255
            cv2.imwrite(out_skel_path,skel)
@torch.no_grad()
def pre_soft_skeletonize(base_path,output_path,batch_size=10,k=25):
    parts = ["train","val","test"]
    os.makedirs(os.path.join(output_path , "skels_soft"),exist_ok=True)
    for part in parts:
        mask_base_path = os.path.join(base_path,"labels",part)
        os.makedirs(os.path.join(output_path , "skels_soft",part),exist_ok=True)

        mask_list = os.listdir(mask_base_path)
        mask_buffer = []
        name_buffer = []
        for i,mask_name in enumerate(tqdm(mask_list)):
            name = Path(mask_name).stem
            mask_path = os.path.join(mask_base_path,mask_name)
            mask = zarr.load(str(mask_path))
            mask = (mask!=0).astype(np.float32)

            mask = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0)
            mask_buffer.append(mask)
            name_buffer.append(name)
            if((i+1)%batch_size==0 or i==len(mask_list)-1):
                mask_buffer = torch.cat(mask_buffer,dim=0).to("cuda")
                skels = soft_skeletonize(mask_buffer,k=k)
                skels = skels.cpu().numpy()
                B = skels.shape[0]
                for i in range(B):
                    skel = skels[i,0].astype(np.uint8)*255
                    o_name = name_buffer[i]
                    out_skel_path = os.path.join(
                        output_path,"skels_soft",part,f"{o_name}.png")
                    cv2.imwrite(out_skel_path,skel)
                mask_buffer = []
                name_buffer = []

@torch.no_grad()
def compute_confution_matrix(data_loader,model,class_maps,
                             output_folder_path=None,
                             draw_plot = True,class_count=26,
                             training_mode = "binary"
                             ):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    conf_mat = torch.zeros((class_count,class_count))
    model.eval()
    for imgs, gt_binary_masks, gt_multi_masks in tqdm(data_loader):
            
        imgs = imgs.to(device)
        if(training_mode=="binary"):
            gt_masks = gt_binary_masks.to(device)
        else:
            gt_masks = gt_multi_masks.to(device)
        with torch.autocast(device_type=device,dtype=torch.float16):
            pred_masks =  model(imgs)
            # pred_targets : B x k x H x W
            # targets : B x H x W

        gt_masks = gt_masks.reshape(-1)
        pred_masks = torch.argmax(pred_masks,dim=1).view(-1) # B x H x W
        encoded_results = (gt_masks*class_count + pred_masks).cpu() # # B x H x W
        counts = torch.bincount(encoded_results,minlength=class_count**2).view(class_count,class_count)
        conf_mat += counts
        
    conf_mat = conf_mat.float() / conf_mat.sum(dim=1,keepdims=True).clamp(min=1)
    conf_mat = conf_mat.numpy()

    if(draw_plot):
        class_names = ["background" for i in range(class_count)]
        for index , name in class_maps.items():
            class_names[index] = name
        plt.figure(figsize=(20,20))
        ax = sns.heatmap(
            conf_mat,
            annot=True,
            fmt=".2f",
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues"
        )
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title("Confusion Matrix")
        plt.tight_layout()
        if(output_folder_path):
            out_path = os.path.join(output_folder_path,"conf_mat.png")
            plt.savefig(out_path)
    return conf_mat
def denormalize(img_norm,channels):
    if(channels==3):
        IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = img_norm * IMAGENET_STD + IMAGENET_MEAN
        img = np.clip(img, 0.0, 1.0)
        img_u8 = (img * 255).astype(np.uint8)
        return img_u8
    

def erode(mask):
    h_pool = -F.max_pool2d(-mask,(3,1),(1,1),(1,0))
    v_pool = -F.max_pool2d(-mask,(1,3),(1,1),(0,1))
    return torch.min(v_pool,h_pool)
def dilate(mask):
    return F.max_pool2d(mask,(3,3),(1,1),(1,1))
def soft_open(mask):
    return dilate(erode(mask))
def soft_skeletonize(I,k=25):
    I_ = soft_open(I)
    S = F.relu(I-I_)
    for i in range(k):
        I = erode(I)
        I_ = soft_open(I)
        S = S + (1-S)*F.relu(I-I_)
    return S

# logger

In [ ]:
colors = np.array([
    (242,  24,  24),   # Red
    (242,  77,  24),   # Red-Orange
    (242, 129,  24),   # Orange
    (242, 181,  24),   # Yellow-Orange
    ( 24, 242, 216),   # Cyan
    (242, 234,  24),   # Yellow
    (146,  24, 242),   # Purple
    (199, 242,  24),   # Yellow-Green
    (146, 242,  24),   # Lime
    ( 94, 242,  24),   # Green
    (242,  24, 181),   # Fuchsia
    ( 42, 242,  24),   # Green (brighter)
    ( 94,  24, 242),   # Violet
    ( 24, 242,  59),   # Spring Green
    (242,  24, 129),   # Pink
    ( 24, 242, 111),   # Aquamarine
    ( 24, 242, 164),   # Turquoise
    ( 24, 164, 242),   # Azure
    (199,  24, 242),   # Magenta
    ( 24, 216, 242),   # Sky Blue
    ( 24, 111, 242),   # Blue
    (242,  24, 234),   # Hot Pink
    ( 24,  59, 242),   # Royal Blue
    ( 42,  24, 242),   # Indigo
    (242,  24,  77),   # Rose
], dtype=np.uint8)


@torch.no_grad()
def save_full_report(recorder,output_base_path,model,valid_loader,normalizer,
                     args,class_map,valid_images,test_transforms,training_mode,
                     class_count,device,name=None,notebook_name="Multi_Main"):
    now = datetime.datetime.now()
    save_folder_name = str(now)
    if(name):
        save_folder_name += f" [{name}]"
    output_folder_path = os.path.join(output_base_path,save_folder_name)

    os.makedirs(output_folder_path,exist_ok=True)
    
    print("Save Model")
    torch.save(model.state_dict(), os.path.join(output_folder_path,"model.pth"))

    print("Saving Memory")
    save_memory(recorder,args,output_folder_path)

    print("Saving All Plots")
    draw_loss_plots(recorder , output_folder_path)
    draw_avg_metric_plots(recorder , output_folder_path)
    draw_all_metric_plots(recorder , output_folder_path)
    compute_confution_matrix(
        data_loader=valid_loader,
        model = model,
        class_maps = class_map,
        draw_plot = True,
        class_count=len(class_map)+1,
        output_folder_path=output_folder_path,
        training_mode = training_mode
    )
    print("Saving Examples")
    draw_examples(
        model = model,
        args = args,
        class_map = class_map,
        valid_images = valid_images,
        test_transforms = test_transforms,
        output_folder_path = output_folder_path,
        device=device,
        training_mode = training_mode,
        normalizer = normalizer
    )

    print("Saving Verbal Results")
    write_verbal_results(
        recorder = recorder,
        training_mode = training_mode,
        output_base_path = output_folder_path
    )

    print("Copying Notebook To Results")

    notebook_out_path = os.path.join(output_folder_path,"notebook.ipynb") 
    shutil.copyfile(f"./{notebook_name}",notebook_out_path )
    print("builfding kaggle project")
    build_kaggle_project(output_folder_path,notebook_name=notebook_name)

def write_verbal_results(recorder,training_mode,output_base_path):
    report = ""
    report_path = os.path.join(output_base_path,"report.txt")
    losses_keys = recorder.losses_keys
    with open("./data/train_count.json","r") as f:
        train_count = json.load(f)

    if(training_mode=="binary"):
        sum_of_counts = sum(train_count.values())
        train_count = {"fg":sum_of_counts}
    for part,data in recorder.metric_avg_list.items():
        report +=f"======= > {part} verbal Report < =======\n"

        dice_list = data["dice"]
        precison_list = data["precision"]
        recall_list = data["recall"]

        best_idx = int(np.argmax(dice_list))
        
        best_dice = dice_list[best_idx]
        best_precision = precison_list[best_idx]
        best_recall = recall_list[best_idx]


        report += (
            f"best epoch : [{best_idx+1}]\n"
            f"best dice : [{best_dice}] - best precision : [{best_precision}] - best recall : [{best_recall}] \n"
        )
    
        for loss_name in losses_keys:
            loss_list = recorder.history[part][loss_name]
            best_loss = loss_list[best_idx]

            report += f"bset {loss_name} : [{best_loss}] - "
            
        report+="\n"
        for index , c in recorder.class_maps.items():
            dice = recorder.metric_history[part]["dice"][index][best_idx]
            precision = recorder.metric_history[part]["precision"][index][best_idx]
            recall = recorder.metric_history[part]["recall"][index][best_idx]

            counts = train_count[c]
            report += f"{c} => dice : {dice} - p : {precision} - r : {recall} || train counts : {counts}\n"
        report +="<=><=><=><=><=><=><=><=><=><=><=><=><=><=><=><=><=>\n"
        
    with open(report_path , "w") as f : 
        f.write(report)

def save_memory(recorder,args,output_folder_path):
    history_path = os.path.join(output_folder_path,"loss_history.json")
    full_metric_path = os.path.join(output_folder_path,"full_metric_hostory.json")
    avg_metric_path = os.path.join(output_folder_path,"avg_metric_hostory.json")
    args_path = os.path.join(output_folder_path,"args.json")

    with open(history_path , "w") as f:
        json.dump(recorder.history,f,indent=4)
    with open(full_metric_path , "w") as f:
        json.dump(recorder.metric_history,f,indent=4)
    with open(avg_metric_path , "w") as f:
        json.dump(recorder.metric_avg_list,f,indent=4)
    with open(args_path , "w") as f:
        json.dump(args,f,indent=4)

def draw_loss_plots(recorder,output_folder_path):
    plt.figure(figsize=(15,20))
    
    losses_keys = recorder.losses_keys
    colors = ["g","r","b","y","orange"]
    colors_per_class = {}

    for i,loss_name in enumerate(losses_keys):
        colors_per_class[loss_name] = colors[i]

    plt_path =os.path.join(output_folder_path,"loss_plot.png")
    
    for i,part in enumerate(recorder.history):
        plt.subplot(2,1,i+1)
        for loss_name,data in recorder.history[part].items():
            length = len(data)-1
            x = np.arange(length)
            plt.plot(x,data[:-1],color = colors_per_class[loss_name],label=loss_name)
        plt.title(f"{part} loss plot")
        plt.legend()
    plt.savefig(plt_path,dpi=150)

def draw_avg_metric_plots(recorder,output_folder_path):
    plt.figure(figsize=(15,20))
    plt_path =os.path.join(output_folder_path,"avg_metrics.png")
    for i,part in enumerate(recorder.metric_avg_list):
        plt.subplot(2,1,i+1)

        dice_data = recorder.metric_avg_list[part]["dice"]
        precision_data = recorder.metric_avg_list[part]["precision"]
        recall_data = recorder.metric_avg_list[part]["recall"]

        length = len(dice_data)
        x = np.arange(length)
        plt.plot(x,dice_data,color="g",label="dice")
        plt.plot(x,precision_data,color="r",label="precision")
        plt.plot(x,recall_data,color="b",label="recall")
        plt.title(f"{part} avg dice plot")
        plt.legend()
    plt.savefig(plt_path)

def draw_all_metric_plots(recorder,output_folder_path):
    for part in recorder.history: 
        plt_path =os.path.join(output_folder_path,f"{part}_full_metric.png")
        plt.figure(figsize=(30,30))
        for i , class_index  in enumerate(recorder.metric_history[part]["dice"]):
            dice_data = recorder.metric_history[part]["dice"][class_index]
            precision_data = recorder.metric_history[part]["precision"][class_index]
            recall_data = recorder.metric_history[part]["recall"][class_index]

            class_name = recorder.class_maps[class_index]
            plt.subplot(5,5,i+1)
            length = len(dice_data)
            x = np.arange(length)
            plt.plot(x,dice_data,color="g",label="dice")
            plt.plot(x,precision_data,color="r",label="precision")
            plt.plot(x,recall_data,color="b",label="recall")
            plt.title(f"{class_name}")
            plt.legend()

        plt.savefig(plt_path)



@torch.no_grad()
def draw_examples(model,args,class_map,valid_images,
                  test_transforms,output_folder_path,
                  normalizer,device,training_mode,w=6,h=5):
    image_names = {
        "easy":[
            "3","7","12","9","46","66","101","1",
            "10","35","43","49","70","80","84"
        ],
        "normal":[
            "15","23","47","78","2","17","18","26","30",
            "39","45","48","60","99","102"
        ],
        "hard":[
            "8","14","24","58","4","16","20","21","32", 
            "36","63","69","93","107","154"
        ],
        "very hard":[
            "79","132","136","148","161","22","119","44",
            "129","162","163","182","187","153","95"
        ]
    }
    dataloader_set = make_example_datasets(
        valid_images=valid_images,
        image_names=image_names,
        transform=test_transforms,
        normalizer = normalizer
    )
    model.eval()
    for diff , dalaloader in dataloader_set.items():
        plt_path = os.path.join(output_folder_path,f"{diff}_examples.png")
        plt.figure(figsize=(30,30))
        patches = [
            mpatches.Patch(color=np.array(colors[j-1]) / 255.0, label=class_map[j])
            for j in range(1,len(class_map)+1)
        ]
        img_index = 1
        for unchaneged_imgs, imgs, gt_binary_masks, gt_multi_masks  in tqdm(dalaloader):

            if(training_mode=="binary"):
                gt_masks = gt_binary_masks.to(device)
            else:
                gt_masks = gt_multi_masks.to(device)

            imgs = imgs.to(device)
            gt_masks = gt_masks.to(device)

            with torch.autocast(device_type=args["device"],dtype=torch.float16):
                pred_masks =  model(imgs)
            

            pred_masks = pred_masks.cpu().numpy()
            gt_masks = gt_masks.cpu().numpy()

            pred_masks = np.argmax(pred_masks,axis=1) # B x H x W

            
            unchaneged_img = unchaneged_imgs[0].cpu().numpy() # H x W x C

            if(unchaneged_img.shape[-1]==1):
                unchaneged_img = np.concatenate(
                    [unchaneged_img,unchaneged_img,unchaneged_img],
                    axis = -1
                )

            pred_mask = pred_masks[0] #H x W
            gt_mask = gt_masks[0]

            real_annoted = draw_mask(unchaneged_img,gt_mask,args,colors)
            pred_annoted = draw_mask(unchaneged_img,pred_mask,args,colors)
            
            plt.subplot(h,w,img_index)
            plt.imshow(real_annoted)
            plt.title(f"Ground Truth ")
            plt.subplot(h,w,img_index+1)
            plt.imshow(pred_annoted)
            plt.title(f"Predicted ")
            img_index+=2
            if(img_index-1==w):
                plt.legend(
                    handles=patches,
                    bbox_to_anchor=(1.05, 1),
                    loc='upper left',
                    borderaxespad=0.,
                    title="Classes"
                )
        plt.savefig(plt_path)

# preprocessing

In [ ]:
class CLAHE : 
    def __init__(self,clipLimit=2.0,tileGridSize=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    def __call__(self,img):
        enhanced = np.clip(img, 0, 255)
        return self.clahe.apply(enhanced)
class WhiteTopHat:
    def __init__(self,kernel_size = (50, 50),turn_neg = True):
        self.kernel = cv2.getStructuringElement(cv2.MORPH_RECT, kernel_size)
        self.turn_neg = turn_neg
    def __call__(self,img):
        neg_img = cv2.bitwise_not(img)
        tophat_img = cv2.morphologyEx(neg_img, cv2.MORPH_TOPHAT, self.kernel,borderType=cv2.BORDER_REPLICATE)
        # tophat_img = morphology.white_tophat(neg_img, self.kernel) 
        return cv2.subtract(img, tophat_img)
        
# Augementations 
def normalize_xca(image):
    x = image.astype(np.float32, copy=False)
    m = x > 0
    if np.any(m):
        mean = x[m].mean()
        std  = x[m].std()
        x[m] = (x[m] - mean) / (std + 1e-8)
        x[~m] = 0.0
    else:
        x = x / 1.0

    # x = x/255.0

    return {"image":x}

def _clip_like_input(out: np.ndarray, ref: np.ndarray) -> np.ndarray:
    if np.issubdtype(ref.dtype, np.integer):
        info = np.iinfo(ref.dtype)
        out = np.clip(out, info.min, info.max)
        return out.astype(ref.dtype, copy=False)
    return out.astype(ref.dtype, copy=False)

class BrightnessMultiplicativeNNUNet2D(A.ImageOnlyTransform):
    def __init__(self, multiplier_range=(0.70, 1.3), p=0.15, always_apply=False):
        super().__init__(always_apply=always_apply, p=p)
        self.multiplier_range = multiplier_range

    def apply(self, img, **params):
        factor = random.uniform(*self.multiplier_range)
        out = img.astype(np.float32) * factor  # multiply intensities
        return _clip_like_input(out, img)

    def get_transform_init_args_names(self):
        return ("multiplier_range",)

class ContrastAugmentationNNUNet2D(A.ImageOnlyTransform):
    def __init__(self, contrast_range=(0.65, 1.5), p=0.15, always_apply=False):
        super().__init__(always_apply=always_apply, p=p)
        self.contrast_range = contrast_range

    def apply(self, img, **params):
        factor = random.uniform(*self.contrast_range)
        x = img.astype(np.float32)

        if x.ndim == 2:  # grayscale HxW
            mean = x.mean()
            out = (x - mean) * factor + mean
        else:            # HxWxC (also fine for grayscale HxWx1)
            mean = x.mean(axis=(0, 1), keepdims=True)  # per-channel mean
            out = (x - mean) * factor + mean

        return _clip_like_input(out, img)

    def get_transform_init_args_names(self):
        return ("contrast_range",)

# recorder

In [ ]:
class HistoryRecorder:
    def __init__(self,class_maps,losses_keys,class_count = 25):

        self.history = {
            "train":{},
            "valid":{}
        }
        for key in losses_keys:
            self.history["train"][key] = [[]]
            self.history["valid"][key] = [[]]
        self.losses_keys = losses_keys
        self.metric_history={}
        self.class_maps =class_maps
        for part in self.history :
            self.metric_history[part]={"dice":{},"precision":{},"recall":{}}
            for i in range(1,class_count+1):
                self.metric_history[part]["dice"][i]=[]
                self.metric_history[part]["recall"][i]=[]
                self.metric_history[part]["precision"][i]=[]
                
        self.metric_avg_list = {
            "train":{"dice":[],"precision":[],"recall":[]},
            "valid":{"dice":[],"precision":[],"recall":[]}
        }
        self.class_wise_losses = {"train":[],"valid":[]}
        self.class_count = class_count 
        
    def add_losses(self,part,loss_dict):
        for loss_name,loss in loss_dict.items():
            self.history[part][loss_name][-1] += [loss]
        # print(class_wise_loss.shape)

    def add_metrics(self,dice,precision,recall,part):
        dice = dice[1:]
        precision = precision[1:]
        recall = recall[1:]
        for i in range(self.class_count):
            d = dice[i]
            r = recall[i]
            p = precision[i]
            self.metric_history[part]["dice"][i+1].append(d)
            self.metric_history[part]["recall"][i+1].append(r)
            self.metric_history[part]["precision"][i+1].append(p)
    def avg_losses(self,part):
        for key in self.history[part]:
            self.history[part][key][-1] = np.mean(self.history[part][key][-1])
            self.history[part][key].append([])
        
    def print_loss_report(self,part,epoch,avg_first=True):
        if(avg_first):
            self.avg_losses(part)
            
        report = f"{part} ==> epcoh ({epoch})\n"
        co=0
        for loss_name in self.losses_keys:
            loss_list = self.history[part][loss_name]

            loss = loss_list[-2]
            report += f"{loss_name} : {loss}"
            if((co+1)%3==0):
                report += "\n"
            else:
                report+=" - "
            co+=1  
        # if(class_wise==True):
        #     report +="\nclass wise loss :\n"
            
        #     self.class_wise_losses[part] = np.stack(
        #         self.class_wise_losses[part] ,axis=0
        #     ).mean(axis=0).tolist() # (C-1)
        #     for i , class_loss in enumerate(self.class_wise_losses[part]):
        #         c = self.class_maps[i+1]
        #         report += f"{c} : {class_loss}\n"
                
        #     self.class_wise_losses[part]=[] 
        #     report += "=========================="
            
        print(report)
    def print_metrics_report(self,part,epoch,class_wise=False):
        
        report_temp = f"{part} avg metrics for epoch {epoch} :\n"
        report_class_wise_temp = ""
        avg_dice = 0
        avg_precision = 0
        avg_recall = 0
        for index , c in self.class_maps.items():
            dice = self.metric_history[part]["dice"][index][-1]
            precision = self.metric_history[part]["precision"][index][-1]
            recall = self.metric_history[part]["recall"][index][-1]
            avg_dice += dice
            avg_precision += precision
            avg_recall += recall
            if(class_wise):
                report_class_wise_temp += f"{c} => dice : {dice} p : {precision} , r : {recall}\n"
        
        avg_dice = avg_dice/self.class_count
        avg_precision = avg_precision/self.class_count
        avg_recall = avg_recall/self.class_count

        self.metric_avg_list[part]["dice"]+=[avg_dice]
        self.metric_avg_list[part]["precision"]+=[avg_precision]
        self.metric_avg_list[part]["recall"]+=[avg_recall]
        
        report_temp+=f"avg dice : {avg_dice} - avg precision : {avg_precision} - avg recall : {avg_recall}"

        if(class_wise):
            report_temp = report_temp + "\n" +report_class_wise_temp[:-1] #removing last \n
        print(report_temp)

# costume_nnunet_blocks

In [ ]:
def crop_dims():
    pass
class Conv(nn.Module):
    def __init__(self,in_c , out_c,p):
        super(Conv,self).__init__()
        self.layers =  nn.Sequential( 
            nn.Conv2d(
                in_channels = in_c , 
                out_channels = out_c ,
                kernel_size=3, 
                stride = 1 ,
                padding = p
            ),
            nn.InstanceNorm2d(out_c, eps=1e-5, affine=True),
            nn.LeakyReLU(negative_slope=1e-2, inplace=True)
        )
    def forward(self,x):
        return self.layers(x)
class DownsampleConv(nn.Module):
    def __init__(self,in_c , out_c):
        super(DownsampleConv,self).__init__()
        self.layers =  nn.Sequential( 
            nn.Conv2d(
                in_channels = in_c , 
                out_channels = out_c ,
                kernel_size=3, 
                stride = 2,
                padding=1
            ),
            nn.InstanceNorm2d(out_c, eps=1e-5, affine=True),
            nn.LeakyReLU(negative_slope=1e-2, inplace=True)
        )
    def forward(self,x):
        return self.layers(x)
class EncoderBlock(nn.Module):
    def __init__(self,in_c , out_c,p=1):
        super(EncoderBlock,self).__init__()
        self.layers = nn.Sequential(
            Conv(in_c = in_c , out_c = out_c , p=p),
            Conv(in_c = out_c , out_c=out_c ,p=p)
        )
        self.pool =  DownsampleConv(in_c = out_c , out_c=out_c )
    def forward(self,x):
        z = self.layers(x)
        return z , self.pool(z)

class DecoderBlock(nn.Module):
    def __init__(self,in_c ,out_c , f_int_scale,class_count,gate_c = None , attention=False,dsv=False):
        super(DecoderBlock,self).__init__()
        self.dsv=dsv
        self.conv1 = Conv(in_c=in_c , out_c=out_c,p=1)
        self.conv2 = Conv(in_c = out_c , out_c = out_c , p=1)
        self.upsampler = nn.ConvTranspose2d(
            in_channels = out_c , 
            out_channels = out_c//2 ,
            kernel_size=2 ,
            stride=2
        )

        if(self.dsv):
            self.dsv_block = nn.Conv2d(in_channels=out_c,out_channels=class_count,kernel_size=1)
        #in_c * 2 = gate_c
        if(attention):
            self.gate = AttentionGate(gate_in_c=gate_c ,f_int_scale=f_int_scale,skip_in_c=in_c//2)
        self.attention=attention
    def forward(self,x_in,x_skip,x_gate):
        if(self.attention):
            x_skip = self.gate(x_skip , x_gate)
        if(x_in.shape[2] != x_skip.shape[2] or x_in.shape[3] != x_skip.shape[3]):
            x_in = F.interpolate(
                x_in, 
                size=x_skip.shape[2:], 
                mode="bilinear", 
                align_corners=False
            )
        
        x = torch.cat([x_skip,x_in],dim=1)
        z = self.conv1(x)
        gate_z = self.conv2(z)
        upsampled_z = self.upsampler(gate_z)
        if(self.dsv):
            dsv_out = self.dsv_block(gate_z)
            
            if(self.attention):
               
                return upsampled_z , gate_z , dsv_out
            else:
                return upsampled_z , None, dsv_out 
        else:
            if(self.attention):
                return upsampled_z , gate_z , None
            else:
                return upsampled_z , None , None

class BottleNeck(nn.Module):
    def __init__(self,in_c , out_c,p,attention=False):
        super(BottleNeck,self).__init__()
        self.conv1 = Conv(in_c=in_c , out_c=out_c,p=p)
        self.conv2 = Conv(in_c = out_c , out_c = out_c , p=p)
        self.upsampler = nn.ConvTranspose2d(
            in_channels = out_c , 
            out_channels = out_c//2 ,
            kernel_size=2 ,
            stride=2
        )
        self.attention=attention
    def forward(self,x):
        z = self.conv1(x)
        gate_z = self.conv2(z)
        upsampled_z = self.upsampler(gate_z)
        if(self.attention):
            return upsampled_z , gate_z
        return upsampled_z , None

class AttentionGate(nn.Module):
    def __init__(self,gate_in_c,skip_in_c,f_int_scale=2,f_int=None,scaler="sigmoid"):
        super(AttentionGate,self).__init__()
        f_int = min(gate_in_c//f_int_scale,skip_in_c//f_int_scale) if f_int==None else f_int
        f_int = 1 if f_int == 0 else f_int
        self.conv_gate = nn.Conv2d(in_channels = gate_in_c , out_channels = f_int , 
                                   kernel_size = 1)
        self.conv_skip = nn.Conv2d(in_channels = skip_in_c , out_channels = f_int , 
                                   kernel_size = 1)
        self.relu = nn.ReLU(inplace=True)
        self.conv_shrink = nn.Conv2d(in_channels = f_int , out_channels = 1 ,
                                     kernel_size = 1)
        if(scaler =="sigmoid"):
            self.scaler = nn.Sigmoid()    
    def forward(self,x_skip,x_gate):
        x_gate_int = self.conv_gate(x_gate)
        x_skip_int = self.conv_skip(x_skip)
        # x_skip_int = crop_dims(x_gate_int,x_skip_int)
        if x_skip_int.shape[2:] != x_gate_int.shape[2:]:
            x_skip_int = F.interpolate(
                x_skip_int, 
                size=x_gate_int.shape[2:], 
                mode="bilinear", 
                align_corners=False
            )
        
        added_x = x_skip_int + x_gate_int
        relu_x = self.relu(added_x)
        shrinked_x = self.conv_shrink(relu_x)
        sig_x = self.scaler(shrinked_x)
        # padded_x = padd_dims(x_skip , sig_x)
        if sig_x.shape[2:] != x_skip.shape[2:]:
            padded_x = F.interpolate(
                sig_x, 
                size=x_skip.shape[2:], 
                mode="bilinear", 
                align_corners=False
            )

        return padded_x*x_skip

class Head(nn.Module):
    def __init__(self,in_c ,out_c ,class_count ,f_int_scale, 
        gate_c = None , attention=False):

        super(Head,self).__init__()
        self.conv1 = Conv(in_c=in_c , out_c=out_c,p=1)
        self.conv2 = Conv(in_c = out_c , out_c = out_c , p=1)
        self.conv1x1 = nn.Conv2d(
            in_channels = out_c , 
            out_channels = class_count ,
            kernel_size=1
        )
        
        if(attention):
            self.gate = AttentionGate(
                gate_in_c=gate_c , 
                f_int_scale=f_int_scale,
                skip_in_c=in_c//2
            )
        self.attention=attention
    def forward(self,x_in,x_skip,x_gate):
        if(self.attention):
            x_skip = self.gate(x_skip , x_gate)
        if(x_in.shape[2] != x_skip.shape[2] or x_in.shape[3] != x_skip.shape[3]):
            x_in = F.interpolate(
                x_in, 
                size=x_skip.shape[2:], 
                mode="bilinear", 
                align_corners=False
            )
            
        x = torch.cat([x_skip,x_in],dim=1)
        z = self.conv1(x)
        gate_z = self.conv2(z)
        
        class_feature_maps = self.conv1x1(gate_z) 
        
        return class_feature_maps

# nnunet

In [ ]:
class nnUnet(nn.Module):
    def __init__(self,args,encoder_channel_settings=None,decoder_channel_settings=None):
        super(nnUnet,self).__init__()
        
        class_count = args["class_count"]
        attention = args["attention"]
        image_shape = args["image_shape"]
        base_channel = args["base_channel"]
        f_int_scale = args["f_int_scale"]
        max_channels = args["max_channels"]
        input_channels = args["in_c"]
        self.deep_super_vision = args["deep_super_vision"]
        co = args["layer_count"]
        h = image_shape[0]
        w = image_shape[1]
        
        
        for i in range(co):
            w/=2
            h/=2
        
        print(f"number of layers : {co}")

        # create encoder settings 
        if(encoder_channel_settings is None):
            self.encoder_channel_settings = [base_channel]
            for i in range(co-1):
                new_c =min(self.encoder_channel_settings[i]*2,max_channels)
                self.encoder_channel_settings +=[new_c]
        else :
            self.encoder_channel_settings = encoder_channel_settings
        
        # create bottleneck settings
        self.bottle_neck_channel_setting = self.encoder_channel_settings[-1]*2
        # create decoder settings 
        if(decoder_channel_settings is  None):
            self.decoder_channel_settings =[]
            for i in range(co-1):
                self.decoder_channel_settings = [self.encoder_channel_settings[i]*2] +  self.decoder_channel_settings
        else :
            self.decoder_channel_settings = decoder_channel_settings

        
        # build encoder
        self.encoders = nn.ModuleList()
        for i in range(co):
            output_channels = self.encoder_channel_settings[i]
            self.encoders.append(EncoderBlock(in_c=input_channels,out_c=output_channels , p=1))
            input_channels = output_channels
        # build bottleneck

        self.bottle_neck = BottleNeck(in_c = output_channels ,out_c = self.bottle_neck_channel_setting , p=1,attention = attention)
        #build decoder
        input_channels = self.bottle_neck_channel_setting
        self.decoders = []
        for i in range(co-1):
            
            output_channels = self.decoder_channel_settings[i]

            self.decoders = [
                DecoderBlock(
                    in_c = input_channels , 
                    out_c=output_channels , 
                    gate_c = input_channels , 
                    attention = attention,
                    f_int_scale=f_int_scale,
                    dsv = self.deep_super_vision,
                    class_count=class_count
                )] + self.decoders
            
            input_channels = output_channels


        self.decoders = nn.ModuleList(self.decoders)
        self.attention = attention
        
        self.head = Head(
            in_c = input_channels , 
            out_c=input_channels//2 ,
            class_count = class_count,
            gate_c = input_channels , 
            attention = False,
            f_int_scale=f_int_scale
        )
        print("encoder settings : ", self.encoder_channel_settings)
        print("bottle-neck settings : ", self.bottle_neck_channel_setting)
        print("decoder settings : ", self.decoder_channel_settings)
        print("head settings : ",class_count)
    def forward(self,x):
        skips = []
        for encoder in self.encoders : 
            skip , out = encoder(x)
            # print(skip.shape)
            # print(out.shape)
            # print("======")
            skips += [skip]
            x = out
        x_in,gate_in = self.bottle_neck(x)
        
        # print(len(self.decoders))
        for i in range(len(self.decoders) - 1, -1, -1):
            # print("2")
            decoder = self.decoders[i]
            skip = skips[i+1]
            x_out,gate_out,dsv_out = decoder(x_in,skip,gate_in)

            x_in=x_out
            gate_in=gate_out
        # print(x_in.shape)
        output = self.head(
            x_in = x_in,
            x_skip = skips[0],
            x_gate = gate_in
        )
        return output

    # """
    # torch.Size([10, 32, 256, 256])
    # torch.Size([10, 64, 128, 128])
    # torch.Size([10, 128, 64, 64])
    # torch.Size([10, 256, 32, 32])
    # torch.Size([10, 512, 16, 16])
    # torch.Size([10, 512, 8, 8])
    # torch.Size([10, 512, 4, 4])
    # """

# swin_encoder

In [ ]:
class SwinEncoder(nn.Module):
    def __init__(self,args):
        self.confs = {
            "swin_t" : [swin_t,224,96,4,Swin_T_Weights],
            "swin_s" : [swin_s,224,96,4,Swin_S_Weights],
            "swin_b" : [swin_b,224,128,4,Swin_B_Weights],
            "swin_v2_t" : [swin_v2_t,256,96,4,Swin_V2_T_Weights],
            "swin_v2_s" : [swin_v2_s,256,96,4,Swin_V2_S_Weights],
            "swin_v2_b" : [swin_v2_b,256,128,4,Swin_V2_B_Weights]
        }
        super(SwinEncoder,self).__init__()
        swin_head = args["swin_head"]
        swin_type = args["swin_type"]
        class_count = args["class_count"]
        abs_class_count = args["abs_class_count"]
        in_c = args["in_c"]
        deep_super_vision  = args["deep_super_vision"]
        swin_builder , base_img_size , emb_size , depth , weight_fn = self.confs[swin_type]
        self.backbone = swin_builder(weights=weight_fn.IMAGENET1K_V1)
        # print(self.backbone)
        if(in_c!=3):
            self.convert_base_channels(in_c)
        input_h,input_w =args["image_shape"] 
        if(swin_head=="costume"):
            self.head = CostumeHead(
                input_h=input_h,
                input_w=input_w,
                depth=depth,
                emb_size=emb_size,
                class_count = class_count,
                abs_class_count = abs_class_count,
                deep_super_vision=deep_super_vision

            )
    def convert_base_channels(self,in_c):
        old_conv = self.backbone.features[0][0]
        new_conv = nn.Conv2d(
            in_channels=in_c,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=(old_conv.bias is not None)
        )
        with torch.no_grad():
            new_conv.weight[:] = old_conv.weight.mean(dim=1,keepdim=True)
            if old_conv.bias is not None:
                new_conv.bias[:] = old_conv.bias
        self.backbone.features[0][0] = new_conv

    def forward(self,x):
        z = self.backbone.features(x)
        z = self.backbone.norm(z)
        z = self.backbone.permute(z)
        return self.head(z)

class SwinUperNet(nn.Module):
    def __init__(self,args):
        super(SwinUperNet,self).__init__()
        model_name = "openmmlab/upernet-swin-base"
        self.model = AutoModelForSemanticSegmentation.from_pretrained(model_name)
        self.conv1x1 = nn.Conv2d(
            in_channels=150,
            out_channels=args["class_count"],
            kernel_size=1
        )
    def forward(self,x):
        z = self.model(x).logits
        z = self.conv1x1(z)
        return [z]

# swin_blocks

In [ ]:
class CostumeBlock(nn.Module):
    def __init__(self,in_c,out_c):
        super(CostumeBlock,self).__init__()
        self.layer = nn.Sequential(
            nn.ConvTranspose2d(
                in_channels=in_c,
                out_channels=out_c,
                kernel_size=2,
                stride=2,
                bias=True
            ),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(inplace=True),
            
            nn.Conv2d(
                in_channels = out_c , 
                out_channels = out_c ,
                kernel_size=3, 
                stride = 1 ,
                padding = 1
            ),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                in_channels = out_c , 
                out_channels = out_c ,
                kernel_size=3, 
                stride = 1 ,
                padding = 1
            ),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )
    def forward(self,x):
        return self.layer(x)
class CostumeHead(nn.Module):
    def __init__(self,input_h,input_w,depth,emb_size,
                 class_count,abs_class_count,deep_super_vision):
        super(CostumeHead,self).__init__()
        self.layers = nn.ModuleList()
        self.dsv_layers = nn.ModuleList()

        self.input_w = input_w
        self.input_h = input_h
        self.deep_super_vision = deep_super_vision
        in_c = emb_size*(2**(depth-1))
        
        self.stage1 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//2 x H/16 x W/16
        if(self.deep_super_vision):
            self.dsv1 = nn.Conv2d(
                in_channels=in_c//2,
                out_channels=class_count,
                kernel_size=1)
        in_c//=2

        self.stage2 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//4 x H/8 x W/8
        if(self.deep_super_vision):
            self.dsv2 = nn.Conv2d(
                in_channels=in_c//2,
                out_channels=class_count,
                kernel_size=1)
        in_c//=2

        self.stage3 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//8 x H/4 x W/4
        if(self.deep_super_vision):
            self.dsv3 = nn.Conv2d(
                in_channels=in_c//2,
                out_channels=class_count,
                kernel_size=1)
        in_c//=2
        
        self.stage4 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//16 x H/2 x W/2
        if(self.deep_super_vision):
            self.dsv4 = nn.Conv2d(
                in_channels=in_c//2,
                out_channels=class_count,
                kernel_size=1)
        in_c//=2
        

        self.stage5 = CostumeBlock(in_c=in_c,out_c=in_c//2) # C//32 x H x W
        in_c//=2

        self.head = nn.Sequential(
            nn.Conv2d(
                in_channels=in_c, 
                out_channels=in_c//2,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(in_c//2),
            nn.LeakyReLU(),
            

            nn.Conv2d(
                in_channels=in_c//2,
                out_channels=in_c//2,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(in_c//2),
            nn.LeakyReLU(),

            nn.Conv2d(in_channels=in_c//2,out_channels=class_count,kernel_size=1)
        )
        
        
    def forward(self,x):
        z = self.stage1(x)

        z1 = self.stage2(z)
        
        z2 = self.stage3(z1)
        z3 = self.stage4(z2)

        z4 = self.stage5(z3)
        z5 = self.head(z4)

        
        if(self.deep_super_vision):
            z_dsv = self.dsv1(z)
            z1_dsv = self.dsv2(z1)
            z2_dsv = self.dsv3(z2)
            z3_dsv = self.dsv4(z3)
            return [z5,z3_dsv,z2_dsv,z1_dsv,z_dsv]
        else:
            return [z5]
        # torch.Size([8, 512, 24, 24]) 
        # torch.Size([8, 256, 48, 48]) 
        # torch.Size([8, 128, 96, 96]) 
        # torch.Size([8, 64, 192, 192]) 
        # torch.Size([8, 32, 384, 384])

# conv_lstm

In [ ]:
class ConvLSTMBlock(nn.Module):
    def __init__(self,feature_map_size,hidden_size,kernel_size,padding):
        super(ConvLSTMBlock,self).__init__()

        self.conv = nn.Conv2d(
            in_channels=feature_map_size+hidden_size,
            out_channels=hidden_size*4,
            kernel_size=kernel_size,
            padding = padding
        )
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
    def forward(self,x,h,c):
        """
        h : batch_size , d , h' , w'
        c : batch_size , d , h' , w'
        x : batch_size , d , h' , w'
        """
        
        conv_outs = self.conv(torch.cat([x,h],dim=1))
        
        # batch_size , d , h' , w'
        i_conv , f_conv , o_conv , g_conv = torch.chunk(conv_outs,chunks=4 , dim=1)
        
        forget_gate = self.sigmoid(f_conv)
        input_gate = self.sigmoid(i_conv) * self.tanh(g_conv)
        output_gate = self.sigmoid(o_conv)

        c = c*forget_gate + input_gate

        h = output_gate*(self.tanh(c))
        
        return h , c
class ConvLSTM(nn.Module):
    def __init__(self,feature_map_size,hidden_size,kernel_size,padding,stack_size=2,device="cuda"):
        super(ConvLSTM,self).__init__()
        if(hidden_size is None):
            hidden_size = feature_map_size
        models_stack = [
            ConvLSTMBlock(
                feature_map_size=feature_map_size,
                hidden_size=hidden_size,
                kernel_size=kernel_size,
                padding=padding
            )
        ]
        for i in range(stack_size-1):
            models_stack+=[ConvLSTMBlock(
                feature_map_size=hidden_size,
                hidden_size=hidden_size,
                kernel_size=kernel_size,
                padding=padding
            )]
        
        self.models_stack = nn.ModuleList(models_stack)
        self.stack_size=stack_size
        self.hidden_size = hidden_size
        self.feature_map_size = feature_map_size
        self.device=device
    def forward_fn(self, x, H, C):
        model_inp = x
        for i, model in enumerate(self.models_stack):
            h, c = model(model_inp, H[i], C[i])
            H[i], C[i] = h, c
            model_inp = h
        return H, C
    
    def forward(self, x, seq_len):
        B, _, Hh, Ww = x.shape

        H = [x.new_zeros((B, self.hidden_size, Hh, Ww)) for _ in range(self.stack_size)]
        C = [x.new_zeros((B, self.hidden_size, Hh, Ww)) for _ in range(self.stack_size)]

        outputs = []
        for _ in range(seq_len):
            H, C = self.forward_fn(x, H, C)
            outputs.append(H[-1])

        return torch.stack(outputs, dim=0)
class VGG16_Convs(nn.Module):
    def __init__(self):
        super(VGG16_Convs,self).__init__()
        self.vgg = models.vgg16(weights = VGG16_Weights.IMAGENET1K_V1,
                           progress = True).features
    def forward(self,x):
        return self.vgg(x)
    
class FCN_8s(nn.Module):
    def __init__(self,class_count):
        super(FCN_8s,self).__init__()
        self.vgg_convs = VGG16_Convs()

        # Head Layers 
        self.fc6 = nn.Conv2d(in_channels=512,out_channels=1)

class FullConvLSTM(nn.Module):
    def __init__(self,args):
        super(FullConvLSTM,self).__init__()
        feature_map_size = args["feature_map_size"]
        hidden_size = args["hidden_size"]
        padding = args["padding"]
        kernel_size = args["kernel_size"]
        stack_size = args["stack_size"]
        class_count = args["class_count"]
        device = "cuda" if torch.cuda.is_available() else "cpu"

        self.fcn = VGG16_Convs()
        
        self.conv_lstm_model = ConvLSTM(
            feature_map_size=feature_map_size,
            hidden_size=hidden_size,
            kernel_size=kernel_size,
            padding=padding,
            stack_size=stack_size,
            device=device
        )

        self.stop_head = nn.Sequential(
            nn.AdaptiveMaxPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(hidden_size,1)
        )
        self.mask_head = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_size,
                out_channels=hidden_size,
                kernel_size=1,
                bias=True
            ), 
            nn.ConvTranspose2d(hidden_size , hidden_size , kernel_size=4 ,stride=2,padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_size, 48, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(48, class_count, kernel_size=4, stride=2, padding=1)
        )
        self.hidden_size = hidden_size
    def forward(self,x,seq_len):
        """
        inputs : imgs 
            - imgs : (batch_size,C,H,W)
        outputs : stop_label , pred_mask
            - stop_label : (batch_size x seq_len)
            - pred_mask : (batch_size x seq_len,class_count,H,W)
        """
        batch_size , image_channels , h , w = x.shape

        feature_maps = self.fcn(x) #B x d x H//8 x W//8
        
        heads_inputs = self.conv_lstm_model(feature_maps,seq_len) #seq len x B x d x H//8 x W//8
        heads_inputs = heads_inputs.reshape(-1,*heads_inputs.shape[2:])

        stop_label = self.stop_head(heads_inputs) #seq len*B 
        pred_mask = self.mask_head(heads_inputs) #seq len*B x class_count x H x W

        return stop_label , pred_mask

# losses

In [ ]:
class UnetLoss(nn.Module):
    def __init__(self,args,eps = 1e-8):
        super(UnetLoss,self).__init__()
        self.class_count = args["class_count"]
        self.loss_type = args["loss_type"]
        self.mask_ce_weights = args["mask_ce_weights"]
        self.training_mode = args["training_mode"]
        self.eps = eps
        self.args = args
        self.sum_dims = (0,2,3)
        
        self.softmax = nn.Softmax(dim=1)
        ### masks cross-entropy loss (SHARED)
        if(args["mask_ce_weights"] is not None):
            w = torch.tensor(
                args["mask_ce_weights"],
                dtype=torch.float32,
                device="cuda"
            )
            self.ce_fn = nn.CrossEntropyLoss(weight=w)
                
        else :
            self.ce_fn = nn.CrossEntropyLoss()
        ### masks dice loss (SHARED)
        if(self.loss_type=="dice"):
                print("loss is set to dice")
                self.loss_fn = DiceLoss(self.eps,self.sum_dims)
        elif(self.loss_type=="tversky"):
            print("loss is set to tversky")
            self.loss_fn = TverskyLoss(
                eps = self.eps,
                sum_dims= self.sum_dims,
                args = args["tversky_conf"])
        ### JUST BINARY

        if(self.training_mode=="binary"):
            self.cl_dice_loss_fn = CLDiceLoss(
                eps=eps,
                sum_dims=self.sum_dims,
                k=args["cl_dice_conf"]["k"]
            )

    def forward(self,pred_masks,gt_masks):
        if(self.training_mode=="binary"):
            """
            inputs : pred_masks , gt_masks
                - pred_masks : (B,2,H,W)
                - gt_masks : (B,H,W)

            """
            onehot_gt_masks = F.one_hot(gt_masks, num_classes=self.class_count)
            onehot_gt_masks = onehot_gt_masks.permute(0, 3, 1, 2).float()

            pred_probs = self.softmax(pred_masks)

            ### Cross Entropy Loss
            mask_ce_loss = self.ce_fn(pred_masks,gt_masks)
            ### Mask Dice Losses
            fg_pred_probs = pred_probs[:,1:]
            fg_onehot_gt_masks = onehot_gt_masks[:,1:]
            dice_family_loss = self.loss_fn(
                pred_probs = fg_pred_probs,
                gt = fg_onehot_gt_masks
            )
            ### Mask CL-Dice
            cl_dice_loss = self.cl_dice_loss_fn(
                pred_binary_mask = pred_probs[:,1:],
                binary_gt_mask = gt_masks.unsqueeze(1).float()
            )
            ### Recordings
            total_loss = (
                self.args["mask_ce_conf"]["imp_coef"]*mask_ce_loss 
                + self.args[f"{self.loss_type}_conf"]["imp_coef"]*dice_family_loss 
                + self.args["cl_dice_conf"]["imp_coef"]*cl_dice_loss
            )

            loss_dict = {
                "mask CE-Loss" : mask_ce_loss,
                self.loss_type : dice_family_loss,
                "cl dice loss" : cl_dice_loss
            }
        elif(self.training_mode=="multi-class"):
            pass
        
        
        return total_loss , loss_dict

class FocalCrossEntropy(nn.Module):
    def __init__(self,args,eps,mask_ce_weights=None):
        super(FocalCrossEntropy,self).__init__()
        self.eps = eps
        self.f_loss_scale = args["loss_scale"]
        self.f_gamma = args["gamma"]
        if(mask_ce_weights is not None):
            device = "cuda" if torch.cuda.is_available() else "cpu"
            self.mask_ce_weights = torch.tensor(mask_ce_weights).to(device)
        else:
            self.mask_ce_weights = mask_ce_weights

    def forward(self,prob,onehot_mask):
        # prob : (B,C,H,W)
        # onehot_mask : (B,C,H,W)
        # gt_mask = (B,H,W)

        p = (prob*onehot_mask).sum(dim=1) # (B,H,W)
        pt = torch.clamp(p,self.eps,1-self.eps)
        focal_weights = (1-pt)**self.f_gamma
        focal_loss = focal_weights*(torch.log(pt))
        if(self.mask_ce_weights is not None):
            alpha_b = self.mask_ce_weights.view(1, -1, 1, 1).type_as(prob)
            class_w = (alpha_b*onehot_mask).sum(dim=1)
        else :
            class_w = 1.0
        return -self.f_loss_scale*(class_w*focal_loss).mean()

class CLDiceLoss(nn.Module):
    def __init__(self,eps,sum_dims,k=40):
        super(CLDiceLoss,self).__init__()
        self.k=k
        self.eps = eps
        self.sum_dims = (1,2,3)

    def forward(self,pred_binary_mask , binary_gt_mask):

        pred_skel = soft_skeletonize(pred_binary_mask,k=self.k)
        gt_skel = soft_skeletonize(binary_gt_mask,k=self.k)

        t_prec = (pred_skel*binary_gt_mask ).sum(dim=self.sum_dims)/(pred_skel.sum(dim=self.sum_dims) +self.eps)
        t_rec = (gt_skel*pred_binary_mask ).sum(dim=self.sum_dims)/(gt_skel.sum(dim=self.sum_dims) +self.eps)
        
        cldice = 2*((t_prec*t_rec)/(t_prec+t_rec))
        cldice_loss = 1 - cldice.mean()
        return cldice_loss
    
class DiceLoss(nn.Module):
    def __init__(self,eps,sum_dims):
        super(DiceLoss,self).__init__()
        self.eps = eps
        self.sum_dims = sum_dims
    def forward(self,pred_probs,gt):
        tp = (gt * pred_probs).sum(dim=self.sum_dims)
        fp = ((1-gt)*pred_probs).sum(dim=self.sum_dims)
        fn = ((1-pred_probs)*gt).sum(dim=self.sum_dims)
        per_class_dice_score = (2*tp +self.eps)/(2*tp + fp + fn + self.eps)
        # if(present_class is None):
        #     dice_loss = -per_class_dice_score.mean()
        # else:
        #     dice_loss = -per_class_dice_score[present_class].mean()
        dice_loss = -per_class_dice_score.mean()
        return dice_loss

class TverskyLoss(nn.Module):
    def __init__(self, args,eps=1e-6, sum_dims=(0,2,3) ):
        super().__init__()
        self.eps = eps
        self.sum_dims = sum_dims
        self.alpha = args["alpha"]
        self.beta = args["beta"]
        self.gamma = args["gamma"]
        device = "cuda" if torch.cuda.is_available() else "cpu"
        if args["weights"] is not None:
            self.register_buffer("t_alpha", torch.tensor(args["weights"], dtype=torch.float32,device=device))
        else:
            self.t_alpha = None

    def forward(self, pred_probs, gt,batch_size=None,seq_len =None):
        
        if(batch_size is not None):
            pred_probs = pred_probs.reshape((batch_size,seq_len,*pred_probs.shape[1:]))
            gt = gt.reshape((batch_size,seq_len,*gt.shape[1:]))

        gt = gt.float()
        tp = (gt * pred_probs).sum(dim=self.sum_dims)
        fp = ((1 - gt) * pred_probs).sum(dim=self.sum_dims)
        fn = (gt * (1 - pred_probs)).sum(dim=self.sum_dims)
        ti = (tp + self.eps) / (tp + self.alpha * fp + self.beta * fn + self.eps)
        
        loss_c = (1 - ti).clamp_min(0) ** self.gamma
        if(batch_size is not None):
            
            loss_c = loss_c.reshape(-1)
        if self.t_alpha is None:
            return loss_c.mean() 
        w = self.t_alpha
        return (w * loss_c).sum() / (w.sum() + self.eps)

# trainer

In [ ]:
def train_fn(model,imgs,gt_masks,optimizer,loss_fn,scaler,args):
    optimizer.zero_grad()
    loss_dict={}


    with torch.autocast(device_type=args["device"],dtype=torch.float16):
        pred_masks = model(imgs)
        loss , loss_dict = loss_fn(
            pred_masks = pred_masks,
            gt_masks = gt_masks
        )


    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10)

    if random.random() < 0.001:
        with torch.no_grad():
            print("--- Total Norm ---")
            print(total_norm)
            # print("\n--- Gradient norms ---")
            # for name, param in model.named_parameters():
            #     if param.grad is not None:
            #         grad_norm = param.grad.data.norm().item()
            #         print(f"{name:30s}: {grad_norm:.6f}")
            print("----------------------\n")
    scaler.step(optimizer)
    scaler.update()
    
    loss = loss.detach().cpu().item()
    for loss_name in loss_dict:
        loss_dict[loss_name] = loss_dict[loss_name].detach().cpu().item()
    
    loss_dict["total loss"] = loss
    return loss_dict , pred_masks.detach()
    
def trainer(args,recorder,model,optimizer,loss_fn,train_loader,valid_loader,
            lr_sch=None,loss_weights=[1]):
    device = args["device"]
    epcohs = args["epcohs"]
    class_count = args["class_count"]
    full_report_cycle = args["full_report_cycle"]
    training_mode = args["training_mode"]

    scaler = torch.amp.GradScaler(device = device) 
    best_val_dice = float("-inf")
    
    best_model = copy.deepcopy(model)
    for ep in tqdm(range(epcohs)):
        total_TP =  torch.zeros(class_count)
        total_FP = torch.zeros(class_count)
        total_FN = torch.zeros(class_count)
        
        model.train()
        class_wise_report = False
        for imgs, gt_binary_masks, gt_multi_masks in tqdm(train_loader):
            """
            img : (B,C,H,W)
            gt_binary_masks : (B,H,W)
            gt_multi_masks : (B,H,W)
            """
            imgs = imgs.to(device)
            if(training_mode=="binary"):
                gt_masks = gt_binary_masks.to(device)
            else:
                gt_masks = gt_multi_masks.to(device)


            loss_dict , mask_preds = train_fn(
                model = model,
                imgs = imgs,
                gt_masks = gt_masks,
                optimizer = optimizer,
                loss_fn = loss_fn,
                scaler = scaler,
                args = args,

            )


            TP , _ , FP , FN = TP_TN_FP_FN(mask_preds,gt_masks,process_preds=True)
            total_TP += TP
            total_FP += FP
            total_FN += FN
            
            recorder.add_losses("train",loss_dict)

            
        current_lr = [group['lr'] for group in optimizer.param_groups][0]
        print(f"current lr : {current_lr:.4}")
        if(lr_sch is not None):
            lr_sch.step()
        

        dice_score = (2 * total_TP + 1e-8) / (2 * total_TP + total_FP + total_FN + 1e-8)
        precision = total_TP /(total_FP + total_TP + 1e-8) 
        recall = total_TP /(total_FN + total_TP + 1e-8) 
        
        recorder.add_metrics(
            dice_score.tolist(),
            precision.tolist(),
            recall.tolist(),
            part = "train"
        )
        
        recorder.print_loss_report("train",ep)
        recorder.print_metrics_report("train",ep,class_wise=False)
        print("<=>"*20)
        
        if((ep+1)%full_report_cycle==0):
            class_wise_report=True
            
        val_dice = evaluation(
            recorder=recorder,
            model=model,
            loss_fn=loss_fn,
            valid_loader=valid_loader,
            class_wise_report=class_wise_report,
            class_count = class_count,
            epoch=ep,
            device=device,
            training_mode = training_mode
        )
        
        if(val_dice>best_val_dice):
            print(f"New Best! : dice = {val_dice}")
            best_model = copy.deepcopy(model)
            best_val_dice = val_dice
            best_ep = ep + 1
    print(f"best result at epoch {best_ep} with dice {best_val_dice}")
    return best_model



@torch.no_grad()
def evaluation(recorder,model,loss_fn,valid_loader,class_count,training_mode,
               class_wise_report=False,epoch=None,device="cuda"):
    model.eval()
    total_TP = total_TP = torch.zeros(class_count)
    total_FP = torch.zeros(class_count)
    total_FN = torch.zeros(class_count)
    
    for imgs, gt_binary_masks, gt_multi_masks in valid_loader:
        """
        img : (B,C,H,W)
        gt_binary_masks : (B,H,W)
        gt_multi_masks : (B,H,W)
        """
        imgs = imgs.to(device)
        if(training_mode=="binary"):
            gt_masks = gt_binary_masks.to(device)
        else:
            gt_masks = gt_multi_masks.to(device)
        
        with torch.autocast(device_type=device,dtype=torch.float16):
            pred_masks = model(imgs)
            loss , loss_dict = loss_fn(
                pred_masks = pred_masks,
                gt_masks = gt_masks
            )
            
            loss = loss.detach().cpu().item()
            for loss_name in loss_dict:
                loss_dict[loss_name] = loss_dict[loss_name].detach().cpu().item()
        
            loss_dict["total loss"] = loss
        
        
        TP , _ , FP , FN = TP_TN_FP_FN(pred_masks,gt_masks,process_preds=True)
        total_TP += TP
        total_FP += FP
        total_FN += FN
        
        recorder.add_losses("valid",loss_dict)
        
    dice_score = (2 * total_TP + 1e-8) / (2 * total_TP + total_FP + total_FN + 1e-8)
    precision = total_TP /(total_FP + total_TP + 1e-8) 
    recall = total_TP /(total_FN + total_TP + 1e-8) 

    recorder.add_metrics(
        dice_score.tolist(),
        precision.tolist(),
        recall.tolist(),
        part = "valid"
    )
    recorder.print_loss_report("valid",epoch)
    recorder.print_metrics_report("valid",epoch,class_wise=class_wise_report)
    print("-"*60)
    return dice_score[1:].mean().item()

# temp_script

In [ ]:
from trainer import evaluation


# In[2]:


args = {
    "base_path" : "./dataset/syntax/",
    "memory_unet":{
        "layer_count":4,
        "channel_max":256,
        "downsampling":"maxpool",
        "normaliztion":"InstanceNorm2d",
        "activation" : "LeakyReLU",
        "first_layer_out_c" : 64
    },
    "image_shape" : (448,448),
    "in_c":1,

    "training_mode":"binary",
    "report_class_wise":True,
    "attention" : False,
    "batch_size" : 8,
    "num_workers" : 5,
    "device" : "cuda" if torch.cuda.is_available() else "cpu",
    "epcohs":10,
    "f_int_scale" : 2,
    "full_report_cycle" : 1,
    "max_channels":512,
    "loss_type":"tversky",
    "output_base_path" : "./outputs",
    "name" : "binary-tests-tversky",
    "deep_super_vision" : False,
    "mask_ce_weights":None,
    "layer_count":5,
    ### Loss Functions
    "mask_ce_conf":{
        "imp_coef":1.0
    },
    "focal_ce_conf":{
        "gamma":2.0,
        "loss_scale":1.0,
        "imp_coef":1.0
    },
    "tversky_conf":{
        "alpha":0.5,
        "beta":0.5,
        "gamma":2.00,
        "weights":None,
        "imp_coef":1.0
    },
    "cl_dice_conf":{
        "imp_coef":1.0,
        "k":40
    },
    ### Optimizers
    "sgd_conf":{
        "lr" : 0.1,
        "momentum" : 0.99,
        "weight_decay" : 3e-5,
        "nesterov":True
    },
    "adam_conf":{
        "lr" : 0.01,
        "weight_decay" : 3e-5,
    }
}
if(args["training_mode"] == "binary"):
    args["class_count"] = 2 
    class_map = {
        1:"fg"
    }

else:
    args["class_count"] = 26
    class_map = {
        1: '1',2: '2', 3: '3',4: '4',
        5: '5',6: '6',7:'9',8:'7',9:'9a',
        10:'10',11:'8',12:'10a',13:'12',14:'11',
        15:'12a',16:'12b',17:'13',18:'14',19:'14a',
        20:'14b',21:'15',22:'16',
        23: '16a',24: '16b',25: '16c',
    }

# losses_keys = ["total loss","FCE loss",args["loss_type"]]
losses_keys = ["total loss","mask CE-Loss",args["loss_type"],"cl dice loss"]
out_counts = args["layer_count"] if args["deep_super_vision"] else 1
loss_weights = [1/(2**i) for i in range(out_counts)]
loss_weights



# In[3]:


if(args["training_mode"]!="binary"):
    b=0.999

    train_class_counts = [
        1200,374,375,369,303,525,537,
        198,340,70,21,310,1,61,320,129,
        63,305,107,49,127,38,232,43,48,31
    ]

    total = np.sum(train_class_counts)
    mask_ce_weights = np.log(total/np.array(train_class_counts))
    mask_ce_weights = (mask_ce_weights / mask_ce_weights.mean()).tolist()
    mask_ce_weights[0]=0.1

    args["mask_ce_weights"]=mask_ce_weights
    args["mask_ce_weights"]
else:
    args["mask_ce_weights"]=[0.1,1]


# In[4]:


# args["t_alpha"] = [1,0.8]
args["tversky_conf"]["t_alpha"] = None


# In[5]:


train_transforms = A.Compose([
    A.Resize(*args["image_shape"]),
    A.Downscale(
        scale_range=[0.7, 1],
        interpolation_pair={"downscale": 0, "upscale": 0},
        p=0.5
    ),
    A.GaussNoise(
        std_range=[0.0, 0.1],
        mean_range=[0, 0],
        per_channel=True,
        noise_scale_factor=1,
        p=0.5
    ),
    A.GaussianBlur(
        sigma_limit=[1.0,1.5],
        blur_limit = (3,7),
        p=0.5
    ),

    A.OneOf([
        A.ElasticTransform(
            alpha=120, 
            sigma=120 * 0.05, 
            p=1.0
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
        A.OpticalDistortion(distort_limit=0.2, p=1.0),
    ], p=0.5),
    A.Compose([
        A.InvertImg(p=1.0),
        A.RandomGamma(gamma_limit=(70, 150), p=1.0),
        A.InvertImg(p=1.0),
    ], p=0.5),
    BrightnessMultiplicativeNNUNet2D(multiplier_range=(0.70, 1.3), p=0.5),
    ContrastAugmentationNNUNet2D(contrast_range=(0.65, 1.5), p=0.5),
    A.RandomGamma(
        gamma_limit=(90, 120), 
        p=0.5
    ),
    A.Affine(
        scale=(0.7, 1.4),  
        translate_percent=(0, 0),
        rotate=0,               
        shear=0,                 
        fit_output=False, 
        p=0.5
    ),
    A.Rotate(limit=30, p=0.5, fill_mask = 0),
    A.Rotate(limit=-30, p=0.5, fill_mask = 0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    ]
)
test_transforms = A.Compose([
    A.Resize(*args["image_shape"]),
    ]  
)

if(args["in_c"]==1):
    normalizer = normalize_xca
else:
    normalizer = A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
        max_pixel_value=255.0,
    )
train_preprocess = None


# In[6]:


train_images = read_images(base_path = args["base_path"],preprocessor = train_preprocess,part = "train",chosen_labels=[19,25])
valid_images = read_images(base_path = args["base_path"],preprocessor = train_preprocess,part = "val",chosen_labels=[19,25])


# In[7]:


train_ds = TrainUnetDataset(
    transform = train_transforms,
    data = train_images , 
    normalizer = normalizer,
    example_phase=False
)
valid_ds = TrainUnetDataset(
    transform = test_transforms,
    data = valid_images,
    normalizer = normalizer,
    example_phase=False
)
# sampler = WeightedRandomSampler(
#     weights=sampler_weights, 
#     num_samples=len(sampler_weights),
#     replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size = args["batch_size"] ,
    num_workers = args["num_workers"] ,
    pin_memory=True,
    shuffle=True
    # sampler=sampler
)
valid_loader = DataLoader(
    valid_ds,
    batch_size = args["batch_size"]  ,
    num_workers = args["num_workers"] ,
    pin_memory=True,
    shuffle=False
)


# In[8]:


# model = nnUnet(args).to(args["device"])
model = MemoryUnet(args).to(args["device"])

loss_fn = UnetLoss(args)
# optimizer = torch.optim.Adam(
#     model.parameters(), 
#     **args["adam_conf"]
# )
optimizer = torch.optim.SGD(
    model.parameters(),
    **args["sgd_conf"]
)
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
# lr_sch = PolynomialLR(optimizer=optimizer,total_iters=args["epcohs"],power=0.9)
lr_sch = None
recorder = HistoryRecorder(losses_keys=losses_keys,class_maps =class_map,class_count=args["class_count"]-1)


best_model = trainer(
    args=args,
    recorder = recorder,
    model = model,
    optimizer = optimizer,
    loss_fn = loss_fn,
    train_loader = train_loader,
    valid_loader = valid_loader,
    loss_weights=loss_weights,
    lr_sch = lr_sch
)
# best_model = model


# In[9]:


save_full_report(
    recorder= recorder , 
    output_base_path=args["output_base_path"],
    model=best_model,
    valid_loader=valid_loader,
    normalizer = normalizer,
    args=args,
    class_map=class_map,
    name=args["name"],
    valid_images = valid_images,
    test_transforms = test_transforms,
    notebook_name = "Multi_Main.ipynb",
    class_count = args["class_count"],
    device = args["device"],
    training_mode = args["training_mode"]
)


# In[ ]: